In [ ]:
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import re

In [ ]:
def obtener_temp_maxima_filamentos(temperaturas, CF_ranges):
    """
    Obtiene la temperatura máxima para cada filamento.
    
    Parámetros:
    temperaturas (list o np.ndarray): Arreglo 1D con las temperaturas.
    CF_ranges (list): Lista de tuplas (inicio, fin) que delimitan cada filamento.
                              
    Retorna:
    np.ndarray: Arreglo con la temperatura máxima de cada filamento.
    """
    # Convertimos a arreglo de NumPy para mayor rapidez
    temp_array = np.array(temperaturas)
    temperaturas_maximas = []
    
    for inicio, fin in CF_ranges:
        # Extraemos el fragmento correspondiente al filamento.
        # Sumamos 1 al límite final para que el índice "fin" sí se incluya en el corte.
        fragmento = temp_array[inicio:fin+1] 
        
        # Obtenemos el valor máximo de ese fragmento
        temp_max = np.max(fragmento)
        temperaturas_maximas.append(temp_max)
        
    return np.array(temperaturas_maximas)

In [ ]:
def _extraer_fase_y_paso(nombre: str) -> tuple[str, int] | None:
    """
    Extrae fase y paso del nombre de un archivo Estado.

    Formato esperado: ``Estado_{fase}_sim_{N}_paso_{paso}.npz``

    Returns:
        (fase, paso) o None si el nombre no coincide.
    """
    m = re.match(r"Estado_(?P<fase>pp_set|sp_set|pp_reset|sp_reset)_sim_\d+_paso_(?P<paso>\d+)\.npz$", nombre)
    if m is None:
        return None
    return m.group("fase"), int(m.group("paso"))

def recopilar_archivos_fase(directorio: Path, fases_activas: list[str]) -> dict[str, list[tuple[int, Path]]]:
    """
    Escanea un directorio buscando archivos 'Estado_*.npz', extrae su fase y paso,
    y los devuelve agrupados por fase.
    """
    archivos_por_fase: dict[str, list[tuple[int, Path]]] = {}

    for fpath in directorio.glob("Estado_*.npz"):
        resultado = _extraer_fase_y_paso(fpath.name)
        if resultado is None:
            continue

        fase, paso = resultado
        if fase not in fases_activas:
            continue

        archivos_por_fase.setdefault(fase, []).append((paso, fpath))

    return archivos_por_fase


In [ ]:
def obtener_temperatura(sim_path: Path, num_simulacion: int) -> np.ndarray:
    """
    Busca el archivo del paso 0 de una simulación específica y extrae
    su matriz de temperatura.
    """
    # 1. Construimos el nombre exacto del archivo usando f-strings
    nombre_archivo = f"Estado_sp_set_sim_{num_simulacion}_paso_0.npz"

    # 2. Construimos la ruta hacia la subcarpeta 'set' donde está el archivo
    ruta_archivo = sim_path / "set" / nombre_archivo

    # Verificamos que el archivo exista para evitar errores
    if not ruta_archivo.is_file():
        raise FileNotFoundError(f"No se encontró el archivo inicial en: {ruta_archivo}")

    # 3. Cargamos el archivo .npz
    # Usamos un bloque 'with' (context manager) para asegurar que el archivo
    # se cierre correctamente de la memoria después de leerlo.
    with np.load(ruta_archivo) as datos_npz:
        # Extraemos específicamente la matriz llamada "temperatura"
        matriz_temp = datos_npz["temperatura"]

    return matriz_temp


In [ ]:

carpeta_resultados = Path("Results")
CF_ranges = [(0, 59), (60, 119)]

# 1. Creamos una lista para ir acumulando las filas de datos
datos_a_exportar = []

for sim_path in carpeta_resultados.glob("simulation_*"):
    num_sim = int(sim_path.name.split("_")[1])
    try:
        matriz_temperatura = obtener_temperatura(sim_path, num_sim)
        temp_maxima = np.round(obtener_temp_maxima_filamentos(matriz_temperatura, CF_ranges), 3)

        # 2. Añadimos una nueva "fila" con las 3 columnas que necesitas:
        # [Num simulación, Temperatura 1, Temperatura 2]
        datos_a_exportar.append([num_sim, temp_maxima[0], temp_maxima[1]])

        print(f"Simulación {num_sim}: Temperatura máxima por filamento: {temp_maxima}")

    except FileNotFoundError as e:
        print(e)
        continue

# # 3. Guardamos todo en un único fichero FUERA del bucle
# if datos_a_exportar:  # Nos aseguramos de que realmente se extrajeron datos
#     # Convertimos la lista de listas en una matriz bidimensional de NumPy
#     matriz_final = np.array(datos_a_exportar)

#     # Exportamos los datos
#     fich_name = "todas_temp_max_filamentos.txt"
#     np.savetxt(
#         fich_name,
#         matriz_final,
#         fmt=["%d", "%.3f", "%.3f"],  # <-- Pasamos una lista de formatos
#         delimiter="\t",
#         header="Simulacion\tTemp_Filamento_1\tTemp_Filamento_2",
#         comments="",  # Evita que NumPy ponga un símbolo '#' al inicio de la cabecera
#     )
#     print(f"\n¡Proceso finalizado! Los datos se han guardado en '{fich_name}'")
# else:
#     print("\nNo se encontraron datos de simulación para exportar.")


### Estilo de los plots